# Código Sintético — Librería de Patrones Agénticos + Dashboard de Observabilidad

Este notebook acompaña al libro **"Código Sintético: El Arte de Orquestar Agentes de IA"**. Clona el repositorio, instala el paquete `sintetico` (en modo editable, para tener acceso a `demos/` y `tests/`) y ejecuta cada pilar del libro.

In [ ]:
# @title 1. Clonar el repositorio e instalar el paquete { display-mode: "form" }
# Sustituye la URL por la de tu fork/repo si no usas el original.
REPO_URL = "https://github.com/sergioide007/synthetic-code.git"  # <-- edita esto

import os
if not os.path.isdir("codigo_sintetico"):
    get_ipython().system(f"git clone --depth 1 {REPO_URL} codigo_sintetico")
else:
    print("El repo ya está clonado en este entorno.")

get_ipython().run_line_magic("cd", "codigo_sintetico")
get_ipython().system('pip install -e ".[all,dev]" -q')
print("✅ Paquete sintetico instalado en modo editable")

In [ ]:
# @title 2. Verificación rápida de que el paquete importa correctamente { display-mode: "form" }
import sintetico
import trazabilidad

print("sintetico", sintetico.__version__)
print("Modelos registrados:", sorted({cfg.id for cfg in sintetico.MODEL_REGISTRY.values()}))

router = sintetico.ModelRouter()
print("Router de ejemplo ->", router.select_model("Diseña una arquitectura de microservicios"))

## 1. Suite de tests (validación completa de la librería)

Sustituye a `!python patterns_library.py` de la versión anterior de este notebook: ahora la validación es una suite de `pytest` real, con ~85 tests.

In [ ]:
get_ipython().system("pytest tests/ -q --no-header 2>&1 | tail -40")

## 2. Demostración de los 4 pilares (sin API keys, coste $0)

In [ ]:
get_ipython().system("python demos/demo_pilares.py")

## 3. Agente ReAct con trazabilidad estructurada

In [ ]:
get_ipython().system("python demos/demo_trazabilidad.py")

## 4. (Opcional) Demostración de ahorro de costes con API real

Configura tu API key en la celda siguiente antes de ejecutarla. Sin ninguna key configurada, la demo cae automáticamente a un proveedor simulado y lo indica explícitamente — nunca falla silenciosamente ni finge un ahorro que no es real.

In [ ]:
import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # descomenta y pega tu key
# os.environ["OPENAI_API_KEY"] = "sk-..."           # alternativa

get_ipython().system("python demos/demo_ahorro_tokens.py")

## 5. (Opcional) Dashboard de observabilidad estilo Datadog/CloudWatch

El dashboard (`sintetico_api`) es un servidor FastAPI de larga duración, así que en Colab hay que arrancarlo en segundo plano. Esta celda lo hace y expone el puerto con el proxy de Colab.

In [ ]:
# @title Arrancar el dashboard en segundo plano { display-mode: "form" }
get_ipython().system('pip install -e ".[api]" -q')

import subprocess, time, requests

server = subprocess.Popen(
    ["uvicorn", "sintetico_api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

for _ in range(20):
    try:
        r = requests.get("http://127.0.0.1:8000/api/v1/health", timeout=1)
        if r.ok:
            print("✅ Servidor arriba:", r.json())
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("⚠️ El servidor no respondió a tiempo; revisa los logs con server.stdout")

try:
    from google.colab.output import eval_js
    print(eval_js("google.colab.kernel.proxyPort(8000)"))
except ImportError:
    print("No estás en Colab: abre http://localhost:8000/ directamente.")

Cuando termines, detén el servidor con la siguiente celda (si no lo haces, seguirá corriendo hasta que se reinicie el entorno de Colab).

In [ ]:
server.terminate()
print("Servidor detenido")